# 3D Regime Probability & Entropy Surfaces

This notebook experiments with two advanced 3D visualizations:

1. **Regime Probability Surface** - Uses Gaussian Mixture Models to detect market regimes
2. **Entropy Surface** - Measures information entropy across sectors over time

These can help identify:
- Market regime transitions (bull/bear/choppy)
- Information flow and disorder in market structure

In [2]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from scipy.stats import entropy
from sklearn.mixture import GaussianMixture
import warnings
warnings.filterwarnings('ignore')

from data import DataEngine

plt.style.use('dark_background')
print("Libraries loaded.")

Libraries loaded.


## Load Data

In [3]:
# Initialize data engine
engine = DataEngine()

# For a richer analysis, use SPY + sector ETFs
tickers = ['SPY', 'XLF', 'XLK', 'XLE', 'XLV', 'XLI', 'XLP', 'XLY', 'XLB', 'XLU', 'XLRE', 'XLC']

start_date = '2018-01-01'
end_date = '2025-01-01'

prices = engine.get_historical_data(tickers, start_date, end_date)
print(f"Loaded {len(prices)} days of data for {len(prices.columns)} tickers")
prices.head()

AttributeError: 'str' object has no attribute 'to_pydatetime'

In [ ]:
# Compute returns
returns = prices.pct_change().dropna()

# Features: return + rolling volatility (makes regime detection more robust)
spy_returns = returns['SPY']
spy_vol = spy_returns.rolling(20).std()
features = pd.DataFrame({
    'return': spy_returns,
    'vol': spy_vol
}).dropna()

print(f"Feature matrix: {len(features)} observations")

---
## 1. Regime Probability Surface (GMM)

Uses a Gaussian Mixture Model to detect latent market regimes based on returns and volatility:
- **Low Vol (Bull)**: Calm uptrend
- **High Vol (Stress)**: Fear/crisis mode  
- **Transition**: Choppy, uncertain

The 3D surface shows:
- **X-axis**: Time
- **Y-axis**: Regime state (0, 1, 2)
- **Z-axis**: Probability of being in that regime

In [ ]:
def fit_gmm_regime_model(features_df, n_states=3, random_state=42):
    """
    Fits a Gaussian Mixture Model for regime detection.
    
    Args:
        features_df: DataFrame with 'return' and 'vol' columns
        n_states: Number of regimes
        
    Returns:
        model: Fitted GMM model
        posteriors: State probabilities for each time step
    """
    X = features_df.values
    
    model = GaussianMixture(
        n_components=n_states,
        covariance_type='full',
        n_init=10,
        random_state=random_state
    )
    model.fit(X)
    
    # Get posterior probabilities
    posteriors = model.predict_proba(X)
    
    return model, posteriors

# Fit 3-state GMM
model, posteriors = fit_gmm_regime_model(features, n_states=3)

# Reorder states by volatility (using the vol feature's mean per state)
state_vols = [model.means_[i, 1] for i in range(3)]  # vol is index 1
order = np.argsort(state_vols)  # Low vol first
state_labels = ['Low Vol (Bull)', 'Medium Vol', 'High Vol (Stress)']

print("State parameters (ordered by volatility):")
for idx, i in enumerate(order):
    ann_vol = model.means_[i, 1] * np.sqrt(252)
    ann_ret = model.means_[i, 0] * 252
    print(f"  {state_labels[idx]}: Return={ann_ret:+.1%}/yr, Vol={ann_vol:.1%}/yr")

In [ ]:
def plot_regime_probability_surface(dates, posteriors, state_order, state_labels, window=20):
    """
    Creates a 3D surface plot of regime probabilities over time.
    """
    n_states = posteriors.shape[1]
    
    # Smooth probabilities for cleaner surface
    posteriors_smooth = pd.DataFrame(posteriors).rolling(window, min_periods=1).mean().values
    
    # Create grid
    T = len(dates)
    time_idx = np.arange(T)
    state_idx = np.arange(n_states)
    X, Y = np.meshgrid(time_idx, state_idx)
    
    # Reorder posteriors by volatility
    Z = posteriors_smooth[:, state_order].T
    
    # Create figure
    fig = plt.figure(figsize=(16, 10))
    ax = fig.add_subplot(111, projection='3d')
    
    # Color by state
    colors = ['#00ff88', '#ffcc00', '#ff4444']  # Green, Yellow, Red
    
    for i in range(n_states):
        # Plot each regime as a surface ribbon
        ax.plot_surface(
            X[i:i+1, :], 
            Y[i:i+1, :], 
            Z[i:i+1, :],
            color=colors[i],
            alpha=0.7,
            shade=True
        )
    
    ax.set_xlabel('Time', fontsize=12, labelpad=10)
    ax.set_ylabel('Regime', fontsize=12, labelpad=10)
    ax.set_zlabel('Probability', fontsize=12, labelpad=10)
    ax.set_title('Regime Probability Surface (GMM)', fontsize=16, pad=20)
    
    ax.set_yticks([0, 1, 2])
    ax.set_yticklabels(state_labels, fontsize=9)
    
    n_ticks = 8
    tick_idx = np.linspace(0, T-1, n_ticks, dtype=int)
    ax.set_xticks(tick_idx)
    ax.set_xticklabels([dates[i].strftime('%Y-%m') for i in tick_idx], rotation=30, fontsize=9)
    
    ax.set_zlim(0, 1)
    ax.view_init(elev=25, azim=-60)
    
    plt.tight_layout()
    return fig, ax

# Plot the surface
fig, ax = plot_regime_probability_surface(
    features.index, 
    posteriors, 
    order, 
    state_labels,
    window=20
)
plt.show()

In [ ]:
# Alternative: 2D stacked area (often cleaner)
def plot_regime_stacked_area(dates, posteriors, state_order, state_labels):
    fig, ax = plt.subplots(figsize=(16, 6))
    
    probs = pd.DataFrame(posteriors[:, state_order], index=dates, columns=state_labels)
    probs_smooth = probs.rolling(20, min_periods=1).mean()
    
    colors = ['#00ff88', '#ffcc00', '#ff4444']
    ax.stackplot(probs_smooth.index, probs_smooth.T, labels=state_labels, colors=colors, alpha=0.8)
    
    ax.set_xlim(dates[0], dates[-1])
    ax.set_ylim(0, 1)
    ax.set_ylabel('Probability', fontsize=12)
    ax.set_title('Regime Probability Over Time (GMM Posterior)', fontsize=14)
    ax.legend(loc='upper left')
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    return fig, ax

fig, ax = plot_regime_stacked_area(features.index, posteriors, order, state_labels)
plt.show()

---
## 2. Entropy Surface

Shannon entropy measures "disorder" in capital distribution across sectors.

- **High entropy**: Capital spread evenly → diversified, stable
- **Low entropy**: Capital concentrated → herding, fragility

The 3D surface shows:
- **X-axis**: Time
- **Y-axis**: Sector index
- **Z-axis**: Sector contribution to market activity

In [ ]:
def compute_rolling_entropy(returns_df, window=60):
    """
    Computes rolling Shannon entropy of absolute return distribution.
    
    Higher entropy = more evenly distributed (diversified)
    Lower entropy = concentrated (herding)
    """
    sector_rets = returns_df.drop(columns=['SPY'], errors='ignore')
    
    entropy_values = []
    sector_contribs = []
    
    for i in range(len(sector_rets)):
        start_idx = max(0, i - window + 1)
        window_data = sector_rets.iloc[start_idx:i+1]
        
        # Absolute cumulative returns as "weight"
        abs_cum_rets = window_data.abs().sum()
        
        # Normalize to probability
        total = abs_cum_rets.sum()
        if total > 0:
            probs = abs_cum_rets / total
            ent = entropy(probs)
        else:
            probs = pd.Series(1/len(abs_cum_rets), index=abs_cum_rets.index)
            ent = np.log(len(abs_cum_rets))
        
        entropy_values.append(ent)
        sector_contribs.append(probs.values)
    
    entropy_series = pd.Series(entropy_values, index=sector_rets.index)
    sector_contributions = np.array(sector_contribs)
    
    return entropy_series, sector_contributions, sector_rets.columns.tolist()

entropy_series, sector_contribs, sector_names = compute_rolling_entropy(returns, window=60)

max_entropy = np.log(len(sector_names))
print(f"Max possible entropy: {max_entropy:.3f}")
print(f"Current avg entropy: {entropy_series.mean():.3f} ({entropy_series.mean()/max_entropy:.1%} of max)")

In [ ]:
def plot_entropy_surface(dates, sector_contributions, sector_names, entropy_series):
    """
    Creates a 3D surface plot of sector contributions over time.
    """
    n_sectors = len(sector_names)
    T = len(dates)
    
    # Subsample for performance
    step = 5
    dates_sub = dates[::step]
    contribs_sub = sector_contributions[::step, :]
    T_sub = len(dates_sub)
    
    time_idx = np.arange(T_sub)
    sector_idx = np.arange(n_sectors)
    X, Y = np.meshgrid(time_idx, sector_idx)
    Z = contribs_sub.T
    
    fig = plt.figure(figsize=(16, 10))
    ax = fig.add_subplot(111, projection='3d')
    
    surf = ax.plot_surface(
        X, Y, Z,
        cmap='viridis',
        alpha=0.85,
        linewidth=0,
        antialiased=True
    )
    
    ax.set_xlabel('Time', fontsize=12, labelpad=10)
    ax.set_ylabel('Sector', fontsize=12, labelpad=10)
    ax.set_zlabel('Contribution', fontsize=12, labelpad=10)
    ax.set_title('Sector Entropy Surface\n(Height = sector contribution to market activity)', fontsize=14, pad=20)
    
    ax.set_yticks(np.arange(n_sectors))
    ax.set_yticklabels(sector_names, fontsize=8)
    
    n_ticks = 8
    tick_idx = np.linspace(0, T_sub-1, n_ticks, dtype=int)
    ax.set_xticks(tick_idx)
    ax.set_xticklabels([dates_sub[i].strftime('%Y-%m') for i in tick_idx], rotation=30, fontsize=9)
    
    ax.view_init(elev=30, azim=-45)
    fig.colorbar(surf, ax=ax, shrink=0.5, aspect=10, label='Contribution Weight')
    
    plt.tight_layout()
    return fig, ax

fig, ax = plot_entropy_surface(returns.index, sector_contribs, sector_names, entropy_series)
plt.show()

In [ ]:
# Align entropy index with features (which is shorter due to rolling vol)
common_dates = features.index.intersection(returns.index)
entropy_aligned = entropy_series.loc[common_dates]
posteriors_df = pd.DataFrame(posteriors, index=features.index)

def plot_entropy_with_regimes(dates, entropy_series, posteriors_df, state_order, sector_names):
    """
    Overlays entropy with dominant regime.
    """
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(16, 8), sharex=True)
    
    max_ent = np.log(len(sector_names))
    ent_norm = entropy_series / max_ent
    
    ax1.fill_between(dates, ent_norm, alpha=0.5, color='cyan')
    ax1.plot(dates, ent_norm.rolling(20).mean(), color='white', linewidth=2, label='20-day MA')
    ax1.axhline(0.8, color='lime', linestyle='--', alpha=0.5, label='High diversification')
    ax1.axhline(0.6, color='red', linestyle='--', alpha=0.5, label='Concentration warning')
    ax1.set_ylabel('Normalized Entropy', fontsize=12)
    ax1.set_title('Market Entropy (Higher = More Diversified Activity)', fontsize=14)
    ax1.legend(loc='lower right')
    ax1.set_ylim(0.4, 1.0)
    ax1.grid(True, alpha=0.3)
    
    posteriors_aligned = posteriors_df.loc[dates].values
    most_likely = np.argmax(posteriors_aligned[:, state_order], axis=1)
    colors_map = {0: '#00ff88', 1: '#ffcc00', 2: '#ff4444'}
    colors = [colors_map[s] for s in most_likely]
    
    ax2.scatter(dates, most_likely, c=colors, s=3, alpha=0.5)
    ax2.set_yticks([0, 1, 2])
    ax2.set_yticklabels(['Low Vol', 'Medium', 'High Vol'])
    ax2.set_ylabel('Dominant Regime', fontsize=12)
    ax2.set_title('GMM Regime Classification', fontsize=14)
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    return fig

fig = plot_entropy_with_regimes(common_dates, entropy_aligned, posteriors_df, order, sector_names)
plt.show()

---
## 3. Combined Analysis: Regime-Entropy Relationship

In [ ]:
# Create analysis dataframe
analysis_df = pd.DataFrame({
    'entropy': entropy_aligned,
    'regime': np.argmax(posteriors_df.loc[common_dates].values[:, order], axis=1),
    'spy_return': returns.loc[common_dates, 'SPY']
})

regime_stats = analysis_df.groupby('regime').agg({
    'entropy': ['mean', 'std'],
    'spy_return': ['mean', 'std', 'count']
})
regime_stats.index = ['Low Vol', 'Medium', 'High Vol']
regime_stats.columns = ['Entropy Mean', 'Entropy Std', 'Return Mean', 'Return Std', 'Days']
regime_stats['Ann Return'] = regime_stats['Return Mean'] * 252
regime_stats['Ann Vol'] = regime_stats['Return Std'] * np.sqrt(252)

print("Regime Summary Statistics:")
display(regime_stats[['Entropy Mean', 'Ann Return', 'Ann Vol', 'Days']])

In [ ]:
# Scatter: Entropy vs Volatility by regime
fig, ax = plt.subplots(figsize=(10, 8))

rolling_vol = returns['SPY'].rolling(20).std() * np.sqrt(252)
rolling_vol_aligned = rolling_vol.loc[common_dates]
max_ent = np.log(len(sector_names))
ent_norm = entropy_aligned / max_ent

colors = ['#00ff88', '#ffcc00', '#ff4444']
regime = analysis_df['regime'].values

for i, label in enumerate(['Low Vol', 'Medium', 'High Vol']):
    mask = regime == i
    ax.scatter(
        ent_norm.values[mask], 
        rolling_vol_aligned.values[mask], 
        c=colors[i], 
        alpha=0.4, 
        s=15, 
        label=label
    )

ax.set_xlabel('Normalized Entropy', fontsize=12)
ax.set_ylabel('20-day Rolling Volatility (annualized)', fontsize=12)
ax.set_title('Entropy vs Volatility by Regime', fontsize=14)
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## Key Insights

From this analysis:

1. **Regime transitions** often precede major market moves
2. **Low entropy** (concentration) tends to appear during stress
3. **High entropy** with low vol = stable diversified market

Consider integrating if:
- Regime detection aligns with known events (COVID crash, 2022 bear)
- Entropy provides early warning before drawdowns